In [ ]:
import numpy as np
import cv2
import rawpy
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage, signal
from skimage import filters
import os
import json


def calculate_mtf50(image):
    """
    Estimate MTF50 from a strong edge.
    Approximate method, mainly for relative comparison.
    """

    # Work in grayscale only
    if len(image.shape) == 3:
        gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    else:
        gray = (image * 255).astype(np.uint8)

    gray = gray.astype(np.float32)

    # Detect prominent edges
    edges = cv2.Canny(gray.astype(np.uint8), 50, 150)

    y_coords, x_coords = np.where(edges > 0)
    if len(y_coords) == 0:
        return 0.0

    # Pick an edge near image center (usually stable)
    center_y, center_x = gray.shape[0] // 2, gray.shape[1] // 2
    distances = (y_coords - center_y)**2 + (x_coords - center_x)**2
    closest_idx = np.argmin(distances)
    edge_y, edge_x = y_coords[closest_idx], x_coords[closest_idx]

    # Extract small patch around the edge
    size = 50
    y1, y2 = max(0, edge_y - size), min(gray.shape[0], edge_y + size)
    x1, x2 = max(0, edge_x - size), min(gray.shape[1], edge_x + size)
    region = gray[y1:y2, x1:x2]

    if region.size == 0:
        return 0.0

    # Average edge response (ESF)
    esf = np.mean(region, axis=0)

    # Differentiate to get LSF
    lsf = np.diff(esf)
    if len(lsf) == 0:
        return 0.0

    # Frequency response
    mtf = np.abs(np.fft.fft(lsf))
    mtf = mtf / (mtf[0] + 1e-10)

    # First frequency where response drops below 50%
    try:
        idx = np.where(mtf[:len(mtf)//2] < 0.5)[0][0]
        mtf50 = idx / len(mtf)
    except:
        mtf50 = 0.5

    return float(mtf50)


def calculate_noise_level(image):
   

    if len(image.shape) == 3:
        gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    else:
        gray = (image * 255).astype(np.uint8)

    gray = gray.astype(np.float32)

    # High-pass residual
    blurred = cv2.GaussianBlur(gray, (5, 5), 2.0)
    high_freq = gray - blurred
    noise_std = np.std(high_freq)

    # Robust estimate using MAD
    median_filtered = cv2.medianBlur(gray.astype(np.uint8), 5).astype(np.float32)
    diff = gray - median_filtered
    mad = np.median(np.abs(diff - np.median(diff)))
    noise_mad = 1.4826 * mad

    # Average both estimates
    return float((noise_std + noise_mad) / 2.0)


def calculate_gradient_sharpness(image):
    """Mean gradient magnitude (edge strength)."""

    if len(image.shape) == 3:
        gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    else:
        gray = (image * 255).astype(np.uint8)

    grad_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

    return float(np.mean(np.sqrt(grad_x**2 + grad_y**2)))


def calculate_laplacian_variance(image):
    """Focus / sharpness indicator."""

    if len(image.shape) == 3:
        gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    else:
        gray = (image * 255).astype(np.uint8)

    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def calculate_edge_density(image):
    """Fraction of pixels detected as edges."""

    if len(image.shape) == 3:
        gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    else:
        gray = (image * 255).astype(np.uint8)

    edges = cv2.Canny(gray, 50, 150)
    return float(np.sum(edges > 0) / edges.size)


def calculate_texture_score(image):
    """Local variance as texture complexity."""

    if len(image.shape) == 3:
        gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    else:
        gray = (image * 255).astype(np.uint8)

    k = 7
    mean = cv2.blur(gray.astype(np.float32), (k, k))
    mean_sq = cv2.blur((gray.astype(np.float32))**2, (k, k))
    return float(np.mean(mean_sq - mean**2))


def calculate_acutance(image):
    """Perceived sharpness based on gradients."""

    if len(image.shape) == 3:
        gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    else:
        gray = (image * 255).astype(np.uint8)

    gx = np.gradient(gray.astype(np.float32), axis=1)
    gy = np.gradient(gray.astype(np.float32), axis=0)

    return float(np.mean(np.abs(gx) + np.abs(gy)))


def calculate_sharpness_quality(image):
    """
    Penalizes extreme edges (ringing / artifacts).
    """

    if len(image.shape) == 3:
        gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    else:
        gray = (image * 255).astype(np.uint8)

    lap = cv2.Laplacian(gray, cv2.CV_64F)
    p95 = np.percentile(np.abs(lap), 95)
    p50 = np.percentile(np.abs(lap), 50)

    return float(min(p50 / p95, 1.0)) if p95 > 0 else 1.0


def calculate_contrast(image):
    

    if len(image.shape) == 3:
        gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    else:
        gray = (image * 255).astype(np.uint8)

    return float(np.std(gray.astype(np.float32) / 255.0))


def apply_unsharp_mask(image, strength=1.0, radius=1.0):
    
    blurred = cv2.GaussianBlur(image, (0, 0), radius)
    return np.clip(cv2.addWeighted(image, 1 + strength, blurred, -strength, 0), 0, 1)


def apply_high_pass_sharpen(image, strength=1.0):
   

    if len(image.shape) == 3:
        result = image.copy()
        for c in range(3):
            blur = cv2.GaussianBlur(image[:, :, c], (0, 0), 2.0)
            result[:, :, c] = np.clip(image[:, :, c] + strength * (image[:, :, c] - blur) * 0.5, 0, 1)
        return result
    else:
        blur = cv2.GaussianBlur(image, (0, 0), 2.0)
        return np.clip(image + strength * (image - blur) * 0.5, 0, 1)


def apply_clahe(image, clip_limit=2.0):
    """Local contrast enhancement."""

    img8 = (image * 255).astype(np.uint8)

    if len(image.shape) == 3:
        lab = cv2.cvtColor(img8, cv2.COLOR_RGB2LAB)
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB).astype(np.float32) / 255.0
    else:
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
        return clahe.apply(img8).astype(np.float32) / 255.0


class DNGQualityProcessor:
   

    def __init__(self, dng_path, jpg_path=None):
        self.dng_path = dng_path
        self.jpg_path = jpg_path
        self.base_image = None
        self.jpg_image = None
        self.quality_levels = {}
        self.metrics = {}

    def load_dng(self):
        # Load RAW with minimal processing
        with rawpy.imread(self.dng_path) as raw:
            rgb = raw.postprocess(
                use_camera_wb=True,
                no_auto_bright=True,
                output_bps=8,
                gamma=(2.222, 4.5)
            )
        self.base_image = rgb.astype(np.float32) / 255.0

    def load_jpg(self):
        # JPG used only as reference
        if self.jpg_path and os.path.exists(self.jpg_path):
            img = cv2.imread(self.jpg_path)
            self.jpg_image = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            return True
        return False

    def generate_quality_levels(self, num_levels=10, start_level=1):
        # Gradually increase enhancement strength
        for i in range(num_levels):
            level = start_level + i
            p = i / max(1, num_levels - 1)

            img = self.base_image.copy()

            if level == 1:
                pass
            elif level <= 4:
                img = apply_unsharp_mask(img, 0.2 + p * 0.4, 0.8 + p * 0.4)
            elif level <= 7:
                img = apply_unsharp_mask(img, 0.6 + p * 0.6, 1.0)
            elif level <= 9:
                img = apply_unsharp_mask(img, 1.0 + p * 0.5, 1.2)
                img = apply_clahe(img, 1.5 + p)
            else:
                img = apply_unsharp_mask(img, 1.5, 1.5)
                img = apply_clahe(img, 3.0)
                img = apply_high_pass_sharpen(img, 0.8)

            self.quality_levels[level] = img

    def calculate_all_metrics(self):
        # Compute metrics for each enhancement level
        for level, img in self.quality_levels.items():
            lap = calculate_laplacian_variance(img)
            noise = calculate_noise_level(img)

            score = 0.7 * (lap / 1000.0) - 0.3 * (noise / 10.0)

            self.metrics[level] = {
                "Level": level,
                "MTF50": round(calculate_mtf50(img), 4),
                "Noise": round(noise, 4),
                "Gradient_Sharpness": round(calculate_gradient_sharpness(img), 2),
                "Laplacian_Variance": round(lap, 2),
                "Edge_Density": round(calculate_edge_density(img), 4),
                "Texture_Score": round(calculate_texture_score(img), 2),
                "Acutance": round(calculate_acutance(img), 2),
                "Sharpness_Quality": round(calculate_sharpness_quality(img), 4),
                "Contrast": round(calculate_contrast(img), 4),
                "Combined_Score": round(score, 4)
            }

    def find_best_level(self):
        # Highest combined score wins
        return max(self.metrics, key=lambda l: self.metrics[l]["Combined_Score"])


if __name__ == "__main__":
    # Run full pipeline on a single DNG
    DNG_PATH = "test.dng"
    JPG_PATH = "test.jpg"

    if not os.path.exists(DNG_PATH):
        print("DNG file not found")
    else:
        proc = DNGQualityProcessor(DNG_PATH, JPG_PATH)
        proc.load_dng()
        proc.load_jpg()
        proc.generate_quality_levels()
        proc.calculate_all_metrics()

        print("Best quality level:", proc.find_best_level())


🔄 Loading DNG: test.dng
✅ DNG loaded: (2014, 3040, 3)

📸 Generating 10 quality levels...
  Level 1: Original (minimal) ✓
  Level 2: Unsharp mask (s=0.24, r=0.84) ✓
  Level 3: Unsharp mask (s=0.29, r=0.89) ✓
  Level 4: Unsharp mask (s=0.33, r=0.93) ✓
  Level 5: Moderate sharpen (s=0.87) ✓
  Level 6: Moderate sharpen (s=0.93) ✓
  Level 7: Moderate sharpen (s=1.00) ✓
  Level 8: Strong + CLAHE (s=1.39, cl=2.28) ✓
  Level 9: Strong + CLAHE (s=1.44, cl=2.39) ✓
  Level 10: Maximum (all techniques) ✓
✅ Generated 10 levels

📊 Calculating ALL metrics for all levels...
  Level 1: MTF=0.0909, Noise=5.02, Texture=143.1, Quality=0.175, Score=0.0770
  Level 2: MTF=0.0909, Noise=5.48, Texture=162.9, Quality=0.170, Score=0.1529
  Level 3: MTF=0.0909, Noise=5.59, Texture=168.4, Quality=0.167, Score=0.1708
  Level 4: MTF=0.0909, Noise=6.45, Texture=174.4, Quality=0.160, Score=0.1678
  Level 5: MTF=0.0909, Noise=7.68, Texture=243.4, Quality=0.162, Score=0.4268
  Level 6: MTF=0.0909, Noise=7.83, Texture=25